In [1]:
# In the previous work done, we have seen how tools and agents work.
# We know that tools can be custom functions, api, etc. And agents are the ones that can use these tools to perform tasks.
# What if we want a  hub for the tools built by us? We could just store them as modules and use them, right?

# But what if we want to share these tools with others? Or maybe we want to have a centralized place where we can share all our tools? 
# This is where the concept of a MCP comes in.

In [2]:
# MCP is Model Context Protocol.
# An app (which can be a mcp client) can send a request to the MCP server (Tools Hub), which can then process the request and return a response.
# So isnt that an API? Yes, it is an API (when used with http or SSE), but it can also be used via stdio when running locally. 
# For example, Claude code runs MCP on stdio.

In [3]:
# My suggestion is to learn the following things about MCP and you should be good to go:
# - Start a MCP Server
# - Create a MCP Client
# - Create a Tool from functions and add it to the MCP Server
# - Create a Tool from an API and add it to the MCP Server
# - Use the Tool via the MCP Client

In [4]:
# pip install mcp mcp[cli]

In [12]:
# Part 1 — Start an MCP Server

# The minimal server. No tools yet — just starts up and is reachable.

In [11]:
# ./exploration/notebooks/mcp_server.py
# This is a simple MCP server that has a single tool that adds two numbers together.
# The file is added; just run it and it will start the MCP server on http://localhost:8000

In [13]:
# You don't run the server directly in the notebook. The **client** spawns it. We will see this in Part 2.

# What `FastMCP` does internally when you call `.run()`:
# - Starts listening on stdin
# - Responds to MCP protocol messages: `initialize`, `tools/list`, `tools/call`
# - Handles the JSON-RPC framing for you

In [14]:
# Part 2 — Create an MCP Client

# The client connects to the server via stdio. It spawns the server process and communicates through it.

In [15]:
import asyncio
import nest_asyncio
nest_asyncio.apply()
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [16]:
async def connect_and_list_tools(server_script: str):
    """
    Spawns the MCP server as a subprocess and lists available tools.
    """
    server_params = StdioServerParameters(
        command="python",
        args=[server_script],
    )

    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            # Handshake: client sends initialize, server responds
            await session.initialize()

            # Ask the server what tools it has
            tools_response = await session.list_tools()
            tools = tools_response.tools

            if not tools:
                print("Server is running but has no tools registered yet.")
            else:
                for tool in tools:
                    print(f"Tool: {tool.name}")
                    print(f"  Description: {tool.description}")
                    print(f"  Input schema: {tool.inputSchema}")



In [18]:
await connect_and_list_tools("mcp_server.py")

Tool: add_numbers
  Description: Adds two numbers together.
  Input schema: {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'add_numbersArguments', 'type': 'object'}


In [19]:
# What happened:
# - `StdioServerParameters` tells the client how to spawn the server process
# - `stdio_client(...)` opens two async streams: one for reading, one for writing
# - `ClientSession` wraps those streams with the MCP protocol
# - `session.initialize()` does the protocol handshake
# - `session.list_tools()` sends `tools/list` to the server and returns the response

# The server had no tools, so we got nothing. Let's fix that.

In [20]:
# Part 3 — Create a Tool from a Function

# A tool is any Python function decorated with `@mcp.tool()`. The SDK reads:
# - Function name → tool name
# - Docstring → tool description (what the LLM reads to decide when to call this)
# - Type hints → JSON input schema
# - Return type → what the tool returns to the caller

In [21]:
# List tools to confirm they're registered
await connect_and_list_tools("mcp_server.py")

Tool: add_numbers
  Description: 
    Adds two numbers and returns the result.
    Use this when the user asks to add or sum two values.
    
  Input schema: {'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'add_numbersArguments', 'type': 'object'}
Tool: reverse_string
  Description: 
    Reverses the characters in a string.
    Use this when the user wants to reverse text.
    
  Input schema: {'properties': {'text': {'title': 'Text', 'type': 'string'}}, 'required': ['text'], 'title': 'reverse_stringArguments', 'type': 'object'}
Tool: calculate_bmi
  Description: 
    Calculates BMI given weight in kilograms and height in meters.
    Returns the BMI value and the category (Underweight, Normal, Overweight, Obese).
    
  Input schema: {'properties': {'weight_kg': {'title': 'Weight Kg', 'type': 'number'}, 'height_m': {'title': 'Height M', 'type': 'number'}}, 'required': ['weight_kg', 'height_m'], 'title': 'ca

In [22]:
# We see all three tools with their descriptions and input schemas auto-generated from the type hints.

# **Key point:** The docstring is the tool description the LLM uses to decide *when* to call this tool. 
# Write it from the LLM's perspective — describe when and why to use it, not how it works internally.

In [23]:
async def run_all_tools():
    server_params = StdioServerParameters(
        command="python",
        args=["mcp_server.py"],
    )

    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            # --- Tool 1: add_numbers ---
            print("=" * 50)
            print("Calling: add_numbers(a=12.5, b=7.3)")
            result = await session.call_tool("add_numbers", {"a": 12.5, "b": 7.3})
            # result.content is a list of content blocks
            print(f"Result: {result.content[0].text}")

            # --- Tool 2: reverse_string ---
            print("=" * 50)
            print("Calling: reverse_string(text='hello mcp')")
            result = await session.call_tool("reverse_string", {"text": "hello mcp"})
            print(f"Result: {result.content[0].text}")

            # --- Tool 3: calculate_bmi ---
            print("=" * 50)
            print("Calling: calculate_bmi(weight_kg=70, height_m=1.75)")
            result = await session.call_tool("calculate_bmi", {"weight_kg": 70, "height_m": 1.75})
            print(f"Result: {result.content[0].text}")

            # --- Tool 4: get_current_weather (API tool) ---
            # Coordinates for Mumbai
            print("=" * 50)
            print("Calling: get_current_weather(latitude=19.07, longitude=72.87) — Mumbai")
            result = await session.call_tool(
                "get_current_weather",
                {"latitude": 19.07, "longitude": 72.87}
            )
            print(f"Result: {result.content[0].text}")


await run_all_tools()

Calling: add_numbers(a=12.5, b=7.3)
Result: 19.8
Calling: reverse_string(text='hello mcp')
Result: pcm olleh
Calling: calculate_bmi(weight_kg=70, height_m=1.75)
Result: {
  "bmi": 22.86,
  "category": "Normal"
}
Calling: get_current_weather(latitude=19.07, longitude=72.87) — Mumbai
Result: {
  "temperature_celsius": 29.1,
  "wind_speed_kmh": 11.0,
  "weather_code": 1,
  "latitude": 19.07,
  "longitude": 72.87
}
